# Unit Tests — test_train_model.py

Tests every model's train and test functions using small synthetic data.
Run each cell to verify all models work correctly.

## Setup

In [22]:
# NOTE: If kagglehub is not installed, run one of these in a Jupyter cell:
#   !pip install kagglehub
#   !pip3 install kagglehub
#   import sys; !{sys.executable} -m pip install kagglehub

import sys
!{sys.executable} -m pip install kagglehub


In [23]:
import numpy as np
import pandas as pd
import sys, os
sys.path.insert(0, '.')
from src.train_model import (
    train_trivial, test_trivial,
    train_ridge, test_ridge,
    train_rf, test_rf,
    train_xgboost_default, test_xgboost_default,
    train_xgboost_tuned, test_xgboost_tuned,
    train_stacking, test_stacking,
    run_ablation,
)
print("All imports successful")

All imports successful


## Create Synthetic Model Data

In [24]:
# Small fake train/val/test data (1000/200/200 rows, 10 features)
np.random.seed(42)
n_features = 10
feature_names = [f'feature_{i:02d}' for i in range(n_features)]

def make_df(n_rows):
    X = pd.DataFrame(np.random.randn(n_rows, n_features), columns=feature_names)
    y = pd.Series(np.random.randn(n_rows), name='responder_6')
    w = pd.Series(np.random.uniform(0.5, 2.0, n_rows), name='weight')
    return X, y, w

X_train, y_train, w_train = make_df(1000)
X_val, y_val, w_val = make_df(200)
X_test, y_test, w_test = make_df(200)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (1000, 10), Val: (200, 10), Test: (200, 10)


## Test: Trivial Baseline

In [25]:
train_mean, val_rmse, val_r2 = train_trivial(y_train, y_val, w_val)
test_rmse, test_r2 = test_trivial(train_mean, y_test, w_test)

assert isinstance(train_mean, float), "FAIL: train_mean should be float"
assert test_rmse > 0, "FAIL: test_rmse should be positive"
assert abs(val_r2) < 0.1, "FAIL: trivial R^2 should be near 0"
print("PASSED — trivial baseline returns correct types, R^2 near 0")

=== Trivial Baseline ===
Training mean: -0.039909
[Validate] RMSE: 1.0845 | R^2: -0.0112
[Test]     RMSE: 0.9985 | R^2: -0.0004

PASSED — trivial baseline returns correct types, R^2 near 0


## Test: Ridge Regression

In [26]:
model, scaler, val_rmse, val_r2 = train_ridge(
    X_train, y_train, w_train, X_val, y_val, w_val)
preds, test_rmse, test_r2 = test_ridge(
    model, scaler, X_test, y_test, w_test)

assert len(preds) == len(y_test), f"FAIL: expected {len(y_test)} predictions, got {len(preds)}"
assert test_rmse > 0, "FAIL: test_rmse should be positive"
print(f"PASSED — Ridge: {len(preds)} predictions, RMSE {test_rmse:.4f}")

=== Ridge Regression ===
[Train]    Time: 0.0s
[Validate] RMSE: 1.0854 | R^2: -0.0129
[Test]     RMSE: 1.0029 | R^2: -0.0092

PASSED — Ridge: 200 predictions, RMSE 1.0029


## Test: Random Forest

In [27]:
model, val_rmse, val_r2 = train_rf(
    X_train, y_train, w_train, X_val, y_val, w_val,
    n_estimators=10, max_depth=5, min_samples_leaf=10)
preds, test_rmse, test_r2 = test_rf(
    model, X_test, y_test, w_test)

assert len(preds) == len(y_test), f"FAIL: expected {len(y_test)} predictions, got {len(preds)}"
assert test_rmse > 0, "FAIL: test_rmse should be positive"
print(f"PASSED — Random Forest: {len(preds)} predictions, RMSE {test_rmse:.4f}")

=== Random Forest ===
[Train]    Time: 0.0s
[Validate] RMSE: 1.0905 | R^2: -0.0224
[Test]     RMSE: 1.0164 | R^2: -0.0366

PASSED — Random Forest: 200 predictions, RMSE 1.0164


## Test: XGBoost Default

In [28]:
model, val_rmse, val_r2 = train_xgboost_default(
    X_train, y_train, w_train, X_val, y_val, w_val,
    n_estimators=10, max_depth=3)
preds, test_rmse, test_r2 = test_xgboost_default(
    model, X_test, y_test, w_test)

assert len(preds) == len(y_test), f"FAIL: expected {len(y_test)} predictions, got {len(preds)}"
assert test_rmse > 0, "FAIL: test_rmse should be positive"
print(f"PASSED — XGBoost Default: {len(preds)} predictions, RMSE {test_rmse:.4f}")

=== XGBoost (Default) ===
[Train]    Time: 0.0s
[Validate] RMSE: 1.0852 | R^2: -0.0126
[Test]     RMSE: 1.0080 | R^2: -0.0195

PASSED — XGBoost Default: 200 predictions, RMSE 1.0080


## Test: XGBoost Tuned (with early stopping)

In [29]:
model, val_rmse, val_r2 = train_xgboost_tuned(
    X_train, y_train, w_train, X_val, y_val, w_val,
    n_estimators=500, max_depth=3, early_stopping_rounds=10)
preds, test_rmse, test_r2 = test_xgboost_tuned(
    model, X_test, y_test, w_test)

assert len(preds) == len(y_test), f"FAIL: expected {len(y_test)} predictions, got {len(preds)}"
assert model.best_iteration < 500, f"FAIL: early stopping should kick in before 500, got {model.best_iteration}"
print(f"PASSED — XGBoost Tuned: {len(preds)} predictions, stopped at iteration {model.best_iteration}")

=== XGBoost (Tuned) ===
Best iteration: 18
[Train]    Time: 0.0s
[Validate] RMSE: 1.0791 | R^2: -0.0012
[Test]     RMSE: 0.9983 | R^2: -0.0001

PASSED — XGBoost Tuned: 200 predictions, stopped at iteration 18


## Test: Stacking Ensemble

In [30]:
# Train base models first
ridge_model, ridge_scaler, _, _ = train_ridge(
    X_train, y_train, w_train, X_val, y_val, w_val)
xgbt_model, _, _ = train_xgboost_tuned(
    X_train, y_train, w_train, X_val, y_val, w_val,
    n_estimators=50, max_depth=3, early_stopping_rounds=10)

# Train and test stacking
meta_model, meta_scaler, val_rmse, val_r2 = train_stacking(
    ridge_model, ridge_scaler, xgbt_model,
    X_train, y_train, w_train, X_val, y_val, w_val)
preds, test_rmse, test_r2 = test_stacking(
    meta_model, meta_scaler, ridge_model, ridge_scaler, xgbt_model,
    X_test, y_test, w_test)

assert len(preds) == len(y_test), f"FAIL: expected {len(y_test)} predictions, got {len(preds)}"
assert len(meta_model.coef_) == 2, f"FAIL: meta-model should have 2 coefficients, got {len(meta_model.coef_)}"
print(f"PASSED — Stacking: {len(preds)} predictions, 2 meta-model coefficients")

=== Ridge Regression ===
[Train]    Time: 0.0s
[Validate] RMSE: 1.0854 | R^2: -0.0129
=== XGBoost (Tuned) ===
Best iteration: 18
[Train]    Time: 0.1s
[Validate] RMSE: 1.0791 | R^2: -0.0012
=== Stacking Ensemble (Ridge + XGBoost Tuned -> Meta Ridge) ===
[Train]    Time: 0.0s
[Validate] RMSE: 1.0581 | R^2: 0.0373
  Meta-model coefficients: Ridge=-0.0324, XGBoost=0.2185
[Test]   RMSE: 1.0158 | R^2: -0.0353

PASSED — Stacking: 200 predictions, 2 meta-model coefficients


## Test: Ablation Study

In [31]:
feature_groups = {
    'Group A': feature_names[:5],
    'Group B': feature_names,
}
result = run_ablation(
    feature_groups,
    X_train, y_train, w_train,
    X_val, y_val, w_val,
    X_test, y_test, w_test)

assert len(result) == 2, f"FAIL: expected 2 rows, got {len(result)}"
assert 'Test RMSE' in result.columns, "FAIL: Test RMSE column missing"
print(f"PASSED — Ablation: {len(result)} feature groups tested")
print(result.to_string(index=False))

=== Ablation: Group A (5 features) ===
[Validate] RMSE: 1.0846
[Test]     RMSE: 0.9939 | R^2: 0.0087
Time: 0.2s

=== Ablation: Group B (10 features) ===
[Validate] RMSE: 1.0796
[Test]     RMSE: 0.9952 | R^2: 0.0062
Time: 0.2s

PASSED — Ablation: 2 feature groups tested
Feature Set  Num Features  Val RMSE  Test RMSE  Test R^2
    Group A             5  1.084572   0.993944  0.008680
    Group B            10  1.079613   0.995188  0.006197


## Summary

In [32]:
print("=" * 50)
print("ALL test_train_model.py TESTS PASSED")
print("=" * 50)

ALL test_train_model.py TESTS PASSED
